In [1]:
import pandas as pd
import os
import numpy as np
from IPython.display import display
import pprint
from datetime import datetime




pd.set_option('display.max_rows', 1000)  # Maximum number of rows to display
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)  # Adjust column width for better readability




import os
import pandas as pd
import sys

log_file_path = "streamlit_consumption\streamlit_log.log"


# # Open the log file in append mode ("a")
# sys.stdout = open(log_file_path, "w")
# sys.stderr = open(log_file_path, "w")

# input_dir = os.makedirs("D:\streamlit_ingest", exist_ok=True)
main_folder = "D:/prediction_output"
nested_df_dict = {}


def get_subfolder_name(main_folder, nested_df_dict):
    for root, dirs, files in os.walk(main_folder):
        for file in files:
            if file.endswith(".csv"):
                subfolder_name = os.path.basename(root)  # Get subfolder name
                file_path = os.path.join(root, file)
                df_name = os.path.splitext(file)[0]      # Get filename without .csv

                # Read CSV
                df = pd.read_csv(file_path)

                # Create subfolder dictionary if it doesn't exist
                if subfolder_name not in nested_df_dict:
                    nested_df_dict[subfolder_name] = {}

                # Store the DataFrame inside its subfolder
                nested_df_dict[subfolder_name][df_name] = df
    return nested_df_dict

nested_df_dict = get_subfolder_name(main_folder, nested_df_dict)


# pprint.pprint(nested_df_dict)
print(nested_df_dict.keys())
# print(nested_df_dict)
# print(nested_df_dict["PTS_outputs"].keys())
# df = nested_df_dict["PTS_outputs"]["PTS_output_2025-04-06"]
# display(df)

# Access the inner dictionary

linear_prediction_dict_list = ["PTS_outputs", "AST_outputs", "REB_outputs", "3PM_outputs"]

linear_prediction_df_list = []
for i in linear_prediction_dict_list:
    inner_dict = nested_df_dict[i]

    # Get the last key (in insertion order, which is preserved in Python 3.7+)
    last_key = list(inner_dict.keys())[-1]

    # Get the corresponding DataFrame
    last_df = inner_dict[last_key]

    # Optional: show the key and DataFrame shape
    print(f"Last file: {last_key}, shape: {last_df.shape}")

    linear_prediction_df_list.append(last_df)



import ast

def calculate_score_percentages(row,target):
    # Safely parse the recentgames_PTS list from string to list
    try:
        recent_games = ast.literal_eval(row[f'recentgames_{target}']) if isinstance(row[f'recentgames_{target}'], str) else row[f'recentgames_{target}']
    except Exception:
        return pd.Series([0, 0, 0], index=['First_Pct', 'Second_Pct', 'Third_Pct'])

    if not recent_games or len(recent_games) == 0:
        return pd.Series([0, 0, 0], index=['First_Pct', 'Second_Pct', 'Third_Pct'])

    total_games = len(recent_games)
    
    # Ensure thresholds are integers
    first = int(row[f'{target}_First'])
    second = int(row[f'{target}_Second'])
    third = int(row[f'{target}_Third'])


    

    # Count scores >= threshold
    first_count = sum(int(score) >= first for score in recent_games)
    second_count = sum(int(score) >= second for score in recent_games)
    third_count = sum(int(score) >= third for score in recent_games)

    # Convert to percentages
    first_pct = round((first_count / total_games) * 100, 2)
    second_pct = round((second_count / total_games) * 100, 2)
    third_pct = round((third_count / total_games) * 100, 2)

    if first_pct == 100 :
        first_pct = 95

    if second_pct == 100:
        second_pct = 95
        
    if third_pct == 100:
        third_pct = 95    


    return pd.Series([first_pct, second_pct, third_pct], index=['First_Pct', 'Second_Pct', 'Third_Pct'])





list_of_targets = ["PTS", "AST", "REB", "3PM"]

target_number = [0,1,2,3]

final_processed_dfs = []


for i in target_number:
    
    target = list_of_targets[i]

    df = linear_prediction_df_list[i]

    df[['First', 'Second', 'Third']] = df[target].str.split(' - ', expand=True).astype(int)

    df = df.rename(columns=lambda x: f"{target}_{x}" if x in ['First', 'Second', 'Third'] else x)




    display(df.head(10))

    team_totals = df.groupby('team', sort=False)[[f'{target}_First', f'{target}_Second', f'{target}_Third']].sum()
    team_totals['Total'] = team_totals.sum(axis=1)
    team_totals_reset = team_totals.reset_index()  # keeps 'team' as column, keeps order


    print(team_totals)


    # Reset index to get 'team' as a column
    team_totals_reset = team_totals.reset_index()

    # Sort alphabetically (or however you want)
    # team_totals_reset = team_totals_reset.sort_values(by='team').reset_index(drop=True)

    # Create matchups and determine winner based on 'First'
    matchups = []
    for i in range(0, len(team_totals_reset) - 1, 2):
        team1 = team_totals_reset.iloc[i]
        team2 = team_totals_reset.iloc[i + 1]

        winner = team1['team'] if team1[f'{target}_First'] > team2[f'{target}_First'] else team2['team']
        matchup = f"{team1['team']} vs {team2['team']}"

        matchups.append({
            'Matchup': matchup,
            'Winner': winner,
            'Team1_First': team1[f'{target}_First'],
            'Team2_First': team2[f'{target}_First']
        })

    # Convert to DataFrame
    matchups_df = pd.DataFrame(matchups)

    # Display results
    today_date = datetime.today().strftime('%Y-%m-%d')

    matchup_dir = f"D:/streamlit_ingest/matchup_{target}"

    os.makedirs(matchup_dir, exist_ok=True)

    matchup_file_path = os.path.join(matchup_dir, f"matchup_{today_date}.csv")

    matchups_df.to_csv(matchup_file_path, index=False)



    # display(matchups_df)


    






    # Apply to your dataframe
    df[['First_Pct', 'Second_Pct', 'Third_Pct']] = df.apply(lambda row: calculate_score_percentages(row,target), axis=1)

    first_second_third = df[['Player', 'team' ,f'{target}_First', f'{target}_Second', f'{target}_Third','First_Pct', 'Second_Pct', 'Third_Pct', f'recentgames_{target}']]
    first_second_third[f'recentgames_{target}'] = first_second_third[f'recentgames_{target}'].astype(str).str.replace(r'[\[\]]', '', regex=True)



    first_second_third_dir = f"D:/streamlit_ingest/first_second_third_{target}"

    os.makedirs(first_second_third_dir, exist_ok=True)

    first_second_third_file_path = os.path.join(first_second_third_dir, f"first_second_third_{today_date}.csv")

    first_second_third.to_csv(first_second_third_file_path, index=False)

    display(first_second_third.head(10))
    

    df = df.rename(columns={
    'First_Pct': f'{target}_First_Pct',
    'Second_Pct': f'{target}_Second_Pct',
    'Third_Pct': f'{target}_Third_Pct'})
    df[f'recentgames_{target}'] = df[f'recentgames_{target}'].astype(str).str.replace(r'[\[\]]', '', regex=True)


    # Save the processed df to combine later
    # ['Player', 'team', '3PM', 'cv-1_3PM', 'recentgames_3PM', 'rmse_3PM', '3PM_First', '3PM_Second', '3PM_Third', '3PM_First_Pct', '3PM_Second_Pct', '3PM_Third_Pct']
    final_processed_dfs.append(df)


    # Preview
    # display(df[['Player', 'team' ,f'{target}_First', f'{target}_Second', f'{target}_Third','First_Pct', 'Second_Pct', 'Third_Pct']])


from functools import reduce

# Merge on Player and team
final_merged_df = reduce(
    lambda left, right: pd.merge(left, right, on=['Player', 'team'], how='outer'),
    final_processed_dfs
)


final_merged_df_dir = f"D:/streamlit_ingest/final_merged_df"

os.makedirs(final_merged_df_dir, exist_ok=True)

final_merged_df_file_path = os.path.join(final_merged_df_dir, f"final_merged_df_{today_date}.csv")

final_merged_df.to_csv(final_merged_df_file_path, index=False)

# Final combined dataframe
print(final_merged_df)

dict_keys(['3PM_outputs', 'AST_outputs', 'PTS_outputs', 'REB_outputs'])
Last file: PTS_output_2025-04-13, shape: (485, 6)
Last file: AST_output_2025-04-13, shape: (485, 6)
Last file: REB_output_2025-04-13, shape: (485, 6)
Last file: 3PM_output_2025-04-13, shape: (485, 6)


,Player,team,PTS,cv-1_PTS,recentgames_PTS,rmse_PTS,PTS_First,PTS_Second,PTS_Third
0,Keaton Wallace,ATL,0 - 0 - 12,0.00,"[0, 2, 7, 0, 3, 3, 3, 5, 2, 4]",3.211207e-14,0,0,12
1,Caris LeVert,ATL,3 - 18 - 32,51.87,"[31, 4, 13, 21, 14, 14, 10, 9, 17, 5]",2.913442e-15,3,18,32
2,Dyson Daniels,ATL,0 - 11 - 21,61.82,"[8, 10, 15, 19, 12, 17, 22, 22, 9, 19]",4.000326e-15,0,11,21
3,Zaccharie Risacher,ATL,0 - 12 - 29,40.51,"[12, 38, 8, 16, 12, 9, 9, 36, 5, 18]",2.921746e-15,0,12,29
4,Trae Young,ATL,16 - 30 - 43,73.91,"[36, 24, 28, 23, 16, 25, 29, 19, 29, 19]",4.444180e-15,16,30,43
5,Terance Mann,ATL,0 - 5 - 13,54.98,"[5, 14, 0, 10, 14, 5, 7, 9, 14, 16]",1.899005e-15,0,5,13
6,Clint Capela,ATL,0 - 4 - 12,43.55,"[4, 8, 6, 8, 11, 0, 8, 8, 8, 3]",5.628541e-15,0,4,12
7,Onyeka Okongwu,ATL,0 - 12 - 25,56.13,"[4, 14, 30, 27, 6, 20, 12, 13, 15, 14]",2.233098e-15,0,12,25
8,Mouhamed Gueye,ATL,0 - 6 - 13,36.87,"[10, 5, 6, 8, 6, 11, 0, 9, 4, 6]",6.760941e-15,0,6,13
9,Georges Niang,ATL,0 - 11 - 24,41.33,"[16, 11, 9, 6, 13, 6, 3, 17, 5, 12]",3.067229e-15,0,11,24


      PTS_First  PTS_Second  PTS_Third  Total
team                                         
ATL          48         152        305    505
ORL          26         136        261    423
BOS           8         138        297    443
CHA          11         121        262    394
BKN          20         135        286    441
NYK          18         131        270    419
CLE          35         156        291    482
IND          20         136        273    429
MIA          14         147        303    464
WAS           8         140        313    461
PHI          12         161        367    540
CHI          14         150        313    477
MIL          27         143        286    456
DET          33         120        237    390
HOU          10         148        321    479
DEN          11         135        300    446
MEM          15         132        298    445
DAL           6         157        359    522
MIN          13         125        279    417
UTA          22         172       

C:\Users\mandy\AppData\Local\Temp\ipykernel_26060\2942641469.py:214: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_second_third[f'recentgames_{target}'] = first_second_third[f'recentgames_{target}'].astype(str).str.replace(r'[\[\]]', '', regex=True)


,Player,team,PTS_First,PTS_Second,PTS_Third,First_Pct,Second_Pct,Third_Pct,recentgames_PTS
0,Keaton Wallace,ATL,0,0,12,95.0,95.0,0.0,"0, 2, 7, 0, 3, 3, 3, 5, 2, 4"
1,Caris LeVert,ATL,3,18,32,95.0,20.0,0.0,"31, 4, 13, 21, 14, 14, 10, 9, 17, 5"
2,Dyson Daniels,ATL,0,11,21,95.0,70.0,20.0,"8, 10, 15, 19, 12, 17, 22, 22, 9, 19"
3,Zaccharie Risacher,ATL,0,12,29,95.0,60.0,20.0,"12, 38, 8, 16, 12, 9, 9, 36, 5, 18"
4,Trae Young,ATL,16,30,43,95.0,10.0,0.0,"36, 24, 28, 23, 16, 25, 29, 19, 29, 19"
5,Terance Mann,ATL,0,5,13,95.0,90.0,40.0,"5, 14, 0, 10, 14, 5, 7, 9, 14, 16"
6,Clint Capela,ATL,0,4,12,95.0,80.0,0.0,"4, 8, 6, 8, 11, 0, 8, 8, 8, 3"
7,Onyeka Okongwu,ATL,0,12,25,95.0,80.0,20.0,"4, 14, 30, 27, 6, 20, 12, 13, 15, 14"
8,Mouhamed Gueye,ATL,0,6,13,95.0,70.0,0.0,"10, 5, 6, 8, 6, 11, 0, 9, 4, 6"
9,Georges Niang,ATL,0,11,24,95.0,50.0,0.0,"16, 11, 9, 6, 13, 6, 3, 17, 5, 12"


...

,Player,team,3PM,cv-1_3PM,recentgames_3PM,rmse_3PM,3PM_First,3PM_Second,3PM_Third
0,Keaton Wallace,ATL,0 - 0 - 2,6.90,"[0, 0, 1, 0, 1, 1, 1, 1, 0, 0]",3.945938e-01,0,0,2
1,Caris LeVert,ATL,0 - 2 - 4,29.80,"[4, 0, 2, 3, 1, 1, 0, 1, 2, 1]",8.361503e-15,0,2,4
2,Dyson Daniels,ATL,0 - 0 - 2,38.11,"[0, 0, 0, 2, 1, 3, 0, 0, 1, 3]",1.363064e-08,0,0,2
3,Zaccharie Risacher,ATL,0 - 1 - 4,15.55,"[0, 6, 2, 2, 2, 1, 0, 5, 1, 0]",1.591162e-09,0,1,4
4,Trae Young,ATL,0 - 4 - 7,37.71,"[6, 3, 3, 4, 3, 2, 3, 2, 6, 3]",1.278381e-13,0,4,7
5,Terance Mann,ATL,0 - 1 - 2,12.01,"[1, 1, 0, 1, 2, 1, 0, 1, 2, 2]",1.353168e-09,0,1,2
6,Clint Capela,ATL,0 - 0 - 0,100.00,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",0.000000e+00,0,0,0
7,Onyeka Okongwu,ATL,0 - 1 - 3,57.91,"[0, 1, 4, 2, 0, 1, 0, 1, 1, 2]",3.978337e-14,0,1,3
8,Mouhamed Gueye,ATL,0 - 0 - 1,12.61,"[1, 0, 1, 1, 0, 1, 0, 1, 0, 0]",6.037162e-01,0,0,1
9,Georges Niang,ATL,0 - 2 - 4,41.04,"[4, 3, 1, 2, 3, 2, 1, 4, 1, 3]",2.380788e-14,0,2,4


      3PM_First  3PM_Second  3PM_Third  Total
team                                         
ATL           7          20         45     72
ORL           0          12         34     46
BOS           0          20         52     72
CHA           0          17         38     55
BKN           0          16         41     57
NYK           0          13         31     44
CLE           0          19         42     61
IND           0          15         36     51
MIA           0          18         43     61
WAS           0          14         46     60
PHI           0          13         41     54
CHI           0          19         47     66
MIL           0          17         39     56
DET           0          14         34     48
HOU           0          17         43     60
DEN           0          10         36     46
MEM           0          14         39     53
DAL           0          14         36     50
MIN           0          15         38     53
UTA           0          16       

C:\Users\mandy\AppData\Local\Temp\ipykernel_26060\2942641469.py:214: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  first_second_third[f'recentgames_{target}'] = first_second_third[f'recentgames_{target}'].astype(str).str.replace(r'[\[\]]', '', regex=True)


,Player,team,3PM_First,3PM_Second,3PM_Third,First_Pct,Second_Pct,Third_Pct,recentgames_3PM
0,Keaton Wallace,ATL,0,0,2,95.0,95.0,0.0,"0, 0, 1, 0, 1, 1, 1, 1, 0, 0"
1,Caris LeVert,ATL,0,2,4,95.0,40.0,10.0,"4, 0, 2, 3, 1, 1, 0, 1, 2, 1"
2,Dyson Daniels,ATL,0,0,2,95.0,95.0,30.0,"0, 0, 0, 2, 1, 3, 0, 0, 1, 3"
3,Zaccharie Risacher,ATL,0,1,4,95.0,70.0,20.0,"0, 6, 2, 2, 2, 1, 0, 5, 1, 0"
4,Trae Young,ATL,0,4,7,95.0,30.0,0.0,"6, 3, 3, 4, 3, 2, 3, 2, 6, 3"
5,Terance Mann,ATL,0,1,2,95.0,80.0,30.0,"1, 1, 0, 1, 2, 1, 0, 1, 2, 2"
6,Clint Capela,ATL,0,0,0,95.0,95.0,95.0,"0, 0, 0, 0, 0, 0, 0, 0, 0, 0"
7,Onyeka Okongwu,ATL,0,1,3,95.0,70.0,10.0,"0, 1, 4, 2, 0, 1, 0, 1, 1, 2"
8,Mouhamed Gueye,ATL,0,0,1,95.0,95.0,50.0,"1, 0, 1, 1, 0, 1, 0, 1, 0, 0"
9,Georges Niang,ATL,0,2,4,95.0,70.0,20.0,"4, 3, 1, 2, 3, 2, 1, 4, 1, 3"


                       Player team           PTS  cv-1_PTS                         recentgames_PTS      rmse_PTS  PTS_First  PTS_Second  PTS_Third  PTS_First_Pct  PTS_Second_Pct  PTS_Third_Pct          AST  cv-1_AST                        recentgames_AST      rmse_AST  AST_First  AST_Second  AST_Third  AST_First_Pct  AST_Second_Pct  AST_Third_Pct           REB  cv-1_REB                         recentgames_REB      rmse_REB  REB_First  REB_Second  REB_Third  REB_First_Pct  REB_Second_Pct  REB_Third_Pct        3PM  cv-1_3PM                recentgames_3PM      rmse_3PM  3PM_First  3PM_Second  3PM_Third  3PM_First_Pct  3PM_Second_Pct  3PM_Third_Pct
0                 A.J. Lawson  TOR    0 - 9 - 25     19.07       12, 14, 13, 9, 13, 13, 0, 2, 2, 8  1.111741e-13          0           9         25          95.00           60.00           0.00    0 - 1 - 5     28.82           2, 7, 0, 1, 4, 1, 0, 0, 0, 2  1.765714e+00          0           1          5          95.00           60.00          10.0

In [574]:
import numpy as np

# Step 1: Parse the string
low_str, mean_str, high_str = player_prediction.split(" - ")
low = int(low_str)
mean = int(mean_str)
high = int(high_str) 



# Step 2: Estimate standard deviation from range
std_estimate = (high - low) / 2

# Step 3: Monte Carlo simulation
simulated_outcomes = np.random.normal(loc=mean, scale=std_estimate, size=100000)

# Step 4: Probability of exceeding a threshold, e.g., 20 points
threshold = 24
prob_over_threshold = np.mean(simulated_outcomes >= threshold)

print(f"Probability of scoring ≥ {threshold} points: {prob_over_threshold:.2%}")



NameError: name 'player_prediction' is not defined

In [351]:
import numpy as np
from scipy.stats import norm

def get_most_likely_score(player_prediction: str):
    low_str, mean_str, high_str = player_prediction.split(" - ")
    low, mean, high = int(low_str), int(mean_str), int(high_str)

    std = (high - low) / 2
    simulated = np.random.normal(loc=mean, scale=std, size=10000)
    
    # Round to nearest integer and count frequency
    values, counts = np.unique(simulated.round().astype(int), return_counts=True)
    most_likely_score = values[np.argmax(counts)]
    
    return most_likely_score


In [549]:
get_most_likely_score("16 - 27 - 38")  # Might return 18

27